In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import MDAnalysis as mda
import numpy as np
from MDAnalysis.analysis import rms
from MDAnalysisTests.datafiles import DCD, PSF

from mdadash.backend.analyses.group_rmsd import CustomRMSD, GroupRMSD
from mdadash.backend.kernel.core import BufferedTrajectory

u = mda.Universe(PSF, DCD)

buffer_size = 50
u.trajectory = BufferedTrajectory(u, buffer_size, use_shm=False)

for i in range(buffer_size - 1):
    u.trajectory._prepare_for_next()
    u.trajectory.next()

In [ ]:
a = GroupRMSD()
a.u = u
a.on_post_connect()
groups = a.group_selections.split(",") if a.group_selections.strip() else []

In [4]:
a.rmsd = CustomRMSD(
    a.u,
    a.u.trajectory.ref_u,
    select=a.selection,
    groupselections=groups if len(groups) > 0 else None,
    weights=a._get_weights(),
)
v_new = a._compute_current_frame()
%timeit a._compute_current_frame()

445 μs ± 5.68 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [5]:
a.rmsd = rms.RMSD(
    a.u,
    a.u.trajectory.ref_u,
    select=a.selection,
    groupselections=groups if len(groups) > 0 else None,
    weights=a._get_weights(),
)
v_old = a._compute_current_frame()
%timeit a._compute_current_frame()

2.42 ms ± 36.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [6]:
np.allclose(v_new[2], v_old[2], rtol=1e-05, atol=1e-05)

True